In [1]:
%%capture
!pip install -q "transformers==4.51.3" "peft==0.14.0" "bitsandbytes>=0.49.0" "accelerate>=1.0.0"
!pip install -q "bert-score>=0.3.13" "rouge-score>=0.1.2" scikit-learn iocextract

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BASE_MODEL       = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
HF_REPO          = "Maximuz23/Text-OSINT"
RESULTS_FILE     = "/kaggle/working/result.json"
OUTPUTS_FILE     = "/kaggle/working/output.json"
MAX_SEQ_LENGTH   = 1024
MAX_NEW_TOKENS   = 384
BATCH_SIZE       = 4
MAX_TEST_SAMPLES = 700

TEST_FILE  = "/kaggle/input/datasets/maximuz23/osint-ai-dataset/test.jsonl"
PROBE_FILE = "/kaggle/input/datasets/maximuz23/real-fake-data-check/fake-real.jsonl"

from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map={"": 0},
    token=HF_TOKEN,
    torch_dtype=torch.float16,
)
base.config.use_cache = True
model = PeftModel.from_pretrained(base, HF_REPO, token=HF_TOKEN)
model.eval()

2026-06-13 17:03:55.526035: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781370235.746599      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781370235.815752      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781370236.344136      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781370236.344176      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781370236.344179      23 computation_placer.cc:177] computation placer alr

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/809 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/195M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [4]:
SYSTEM_PROMPT = (
    "You are an expert cybersecurity analyst specializing in Text OSINT and threat "
    "intelligence for red team operations. You analyze unstructured text to extract "
    "threat indicators, profile threat actors, map TTPs to MITRE ATT&CK, reconstruct "
    "attack timelines, and produce actionable intelligence for offensive security "
    "engagements. Work only from the record provided: extract and analyze what is present, "
    "and when a lookup is empty or no record is given, say so plainly instead of inventing "
    "details. Judge by the evidence in the input, not by whether a name looks familiar."
)

def build_chat(prompt):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]

def ask_batch(prompts, use_adapter=True, max_new_tokens=MAX_NEW_TOKENS):
    msgs_list = [build_chat(p) for p in prompts]
    inputs = tokenizer.apply_chat_template(
        msgs_list,
        tokenize=True,
        add_generation_prompt=True,
        padding=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")
    gen_kwargs = dict(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    with torch.no_grad():
        if use_adapter:
            out = model.generate(**gen_kwargs)
        else:
            with model.disable_adapter():
                out = model.generate(**gen_kwargs)
    input_len = inputs.input_ids.shape[1]
    return [tokenizer.decode(out[i][input_len:], skip_special_tokens=True) for i in range(len(prompts))]

In [5]:
import re
import iocextract

REFUSAL_PATTERNS = [
    r"i don'?t have (?:reliable|specific|any|enough|sufficient|direct) (?:information|data|knowledge|context)",
    r"i (?:don'?t|do not) have (?:(?:reliable|specific|any|enough|sufficient|direct|much|the)\s+)?(?:information|data|knowledge|context|details|a profile|record)\b",
    r"i(?:'m| am) not aware of",
    r"i lack (?:reliable |sufficient |the )?(?:information|knowledge|context|data)",
    r"appears to be (?:a )?(?:fictional|made[\- ]up|fake|hypothetical)",
    r"is (?:a )?(?:fictional|fake|hypothetical|made[\- ]up) (?:entry|entity|name|actor|cve|identifier|threat)",
    r"(?:doesn'?t|does not) (?:exist|appear to exist|appear (?:to be )?(?:real|documented|known))",
    r"not a real",
    r"this (?:identifier|cve|hash|actor|name|threat) (?:doesn'?t match|is not (?:known|recognized|documented|real|a real))",
    r"can(?:not|'?t) (?:confirm|verify|find|locate|provide (?:specific|reliable))",
    r"i can'?t (?:provide|confirm|verify|find|locate)",
    r"unable to (?:find|verify|locate|confirm)",
    r"no (?:reliable|public|verified|specific|known)\s+(?:information|record|data)",
    r"no documented (?:information|record|threat|cve|cases|attacks|exploitation|details)",
    r"no public information",
    r"not in (?:my )?(?:training data|dataset|knowledge|database)",
    r"i don'?t recognize",
    r"no authoritative (?:record|information|data|source)",
    r"no matching (?:group|record|entry|cve|identifier)",
    r"returned (?:no matching|nothing|no record)",
    r"lookup returned",
    r"without fabricating",
    r"(?:will not|won'?t|cannot|can'?t|do not|don'?t|refuse to) fabricat\w*",
    r"\bno record of\b",
    r"\bno record for\b",
    r"returns? no data\b",
    r"\bno profile for\b",
    r"can(?:not|'?t) confirm (?:anything|any |the existence)",
    r"could be (?:a )?(?:typo|fictional|vendor[- ]specific|an internal)",
    r"i won'?t speculate",
    r"can(?:not|'?t) (?:query|access) (?:live|real)",
    r"i haven'?t memorized",
    r"no data for\b",
    r"no (?:mitre )?att&ck (?:lookup|record|entry|profile|data)",
    r"found no (?:record|data|entry|match|profile|results?)",
    r"no record found",
    r"\bnot indexed\b",
    r"record (?:is )?not (?:in|listed in|present in|on file|indexed)",
    r"not (?:indexed|on file|present|listed|found) in (?:the )?(?:nvd|kev|mitre)",
]

_REFUSE = re.compile("|".join(REFUSAL_PATTERNS), re.IGNORECASE)

def classify_refusal(text):
    return "refuse" if _REFUSE.search(text) else "answer"


IOC_PATTERNS = {
    "cve":    re.compile(r"\bCVE-\d{4}-\d{4,}\b", re.IGNORECASE),
    "attack": re.compile(r"\bT\d{4}(?:\.\d{3})?\b"),
    "domain": re.compile(
        r"\b[a-z0-9-]+(?:\[?\.\]?[a-z0-9-]+)*\."
        r"(?:ai|app|asia|at|au|bar|be|bet|bid|biz|bond|br|bz|ca|cc|cf|cfd|ch|click|cloud|club|cn|co|com|country|cyou|cz|de|dev|download|edu|es|fi|fr|fun|ga|gdn|gov|gq|gr|hu|icu|in|info|int|io|it|jp|kim|la|life|link|live|loan|me|mil|ml|mn|mobi|monster|name|net|nl|no|nu|onion|online|org|page|pe|pl|pro|pt|pw|racing|rest|review|ro|ru|sbs|se|shop|site|space|store|stream|su|surf|tech|tel|tk|top|tv|uk|us|vip|win|work|world|ws|xyz)\b",
        re.IGNORECASE,
    ),
}

def refang(s):
    return (s.replace("[.]", ".")
             .replace("[:]", ":")
             .replace("hxxps", "https")
             .replace("hxxp", "http"))

_IP_URL = re.compile(r"^https?://(?:\d{1,3}\.){3}\d{1,3}", re.IGNORECASE)

def extract_entities(text):
    out = set()
    for ip in iocextract.extract_ipv4s(text, refang=True):
        out.add(("ipv4", ip.lower()))
    for url in iocextract.extract_urls(text, refang=True):
        u = url.rstrip(".,);:\"'").lower()
        if not _IP_URL.match(u):
            out.add(("url", u))
    for h in iocextract.extract_hashes(text):
        out.add(("hash", h.lower()))
    norm = refang(text)
    for kind in ("cve", "attack", "domain"):
        for m in IOC_PATTERNS[kind].findall(norm):
            out.add((kind, m.rstrip(".,;:!?)\"'").lower()))
    return out

In [6]:
import json, random
random.seed(42)

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

probe    = load_jsonl(PROBE_FILE)
test_all = load_jsonl(TEST_FILE)

if MAX_TEST_SAMPLES and MAX_TEST_SAMPLES < len(test_all):
    test_sample = random.sample(test_all, MAX_TEST_SAMPLES)
else:
    test_sample = test_all

In [7]:
import time

def get_user_msg(r):
    return next(m["content"] for m in r["messages"] if m["role"] == "user")

def get_asst_msg(r):
    return next(m["content"] for m in r["messages"] if m["role"] == "assistant")

def generate_all(prompts, use_adapter, label):
    out, t0, n = [], time.time(), len(prompts)
    for i in range(0, n, BATCH_SIZE):
        out.extend(ask_batch(prompts[i:i+BATCH_SIZE], use_adapter=use_adapter))
        if (i // BATCH_SIZE) % 20 == 0 or i + BATCH_SIZE >= n:
            done = min(i + BATCH_SIZE, n)
            el = time.time() - t0
            rate = done / el if el else 0
            print(f"  [{label}] {done}/{n}  elapsed={el:.0f}s  ETA={(n-done)/rate if rate else 0:.0f}s")
    return out

probe_prompts = [r["user_prompt"] for r in probe]
test_prompts  = [get_user_msg(r) for r in test_sample]

state = {"probe": probe, "test_sample": test_sample}
def save_state():
    with open(OUTPUTS_FILE, "w") as f:
        json.dump(state, f)

for key, prompts, adapter, label in [
    ("probe_ft",   probe_prompts, True,  "probe FT"),
    ("probe_base", probe_prompts, False, "probe BASE"),
    ("test_ft",    test_prompts,  True,  "test FT"),
    ("test_base",  test_prompts,  False, "test BASE"),
]:
    state[key] = generate_all(prompts, adapter, label)
    save_state()

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


  [probe FT] 4/80  elapsed=15s  ETA=276s
  [probe FT] 80/80  elapsed=869s  ETA=0s
  [probe BASE] 4/80  elapsed=16s  ETA=302s
  [probe BASE] 80/80  elapsed=1175s  ETA=0s
  [test FT] 4/700  elapsed=120s  ETA=20875s
  [test FT] 84/700  elapsed=1270s  ETA=9317s
  [test FT] 164/700  elapsed=2423s  ETA=7919s
  [test FT] 244/700  elapsed=3429s  ETA=6409s
  [test FT] 324/700  elapsed=4691s  ETA=5444s
  [test FT] 404/700  elapsed=6073s  ETA=4449s
  [test FT] 484/700  elapsed=7149s  ETA=3190s
  [test FT] 564/700  elapsed=8448s  ETA=2037s
  [test FT] 644/700  elapsed=9744s  ETA=847s
  [test FT] 700/700  elapsed=10540s  ETA=0s
  [test BASE] 4/700  elapsed=116s  ETA=20187s
  [test BASE] 84/700  elapsed=2112s  ETA=15492s
  [test BASE] 164/700  elapsed=4112s  ETA=13438s
  [test BASE] 244/700  elapsed=6054s  ETA=11313s
  [test BASE] 324/700  elapsed=8112s  ETA=9414s
  [test BASE] 404/700  elapsed=10139s  ETA=7429s
  [test BASE] 484/700  elapsed=12087s  ETA=5394s
  [test BASE] 564/700  elapsed=14124s  

In [8]:
from sklearn.metrics import f1_score, precision_score, recall_score
def honesty_metrics(probe, ft_out, base_out, title):
    print(f"\n=== Honesty: {title} ===")
    y_true = [r["expected_behavior"] for r in probe]
    res, preds = {}, {}
    for label, outs in [("base", base_out), ("ft", ft_out)]:
        yp = [classify_refusal(o) for o in outs]
        preds[label] = yp
        res[label] = {
            "accuracy":         sum(t == p for t, p in zip(y_true, yp)) / len(y_true),
            "precision_refuse": precision_score(y_true, yp, pos_label="refuse", zero_division=0),
            "recall_refuse":    recall_score(y_true, yp, pos_label="refuse", zero_division=0),
            "f1_refuse":        f1_score(y_true, yp, pos_label="refuse", zero_division=0),
        }
        h = res[label]
        print(f"  {label.upper():5s} acc={h['accuracy']:.3f} P={h['precision_refuse']:.3f} "
              f"R={h['recall_refuse']:.3f} F1={h['f1_refuse']:.3f}")
    print("  per-category accuracy:")
    for cat in ["fake_actor", "fake_cve", "real_actor", "real_cve"]:
        idx = [i for i, r in enumerate(probe) if r["category"] == cat]
        if not idx: continue
        ab = sum(y_true[i] == preds["base"][i] for i in idx) / len(idx)
        af = sum(y_true[i] == preds["ft"][i]   for i in idx) / len(idx)
        print(f"    {cat:12s} base={ab:.3f} ft={af:.3f}")
    return res

honesty = honesty_metrics(probe, state["probe_ft"], state["probe_base"],
                          "fake-real probe (facts provided)")


=== Honesty: fake-real probe (facts provided) ===
  BASE  acc=0.725 P=1.000 R=0.450 F1=0.621
  FT    acc=0.988 P=1.000 R=0.975 F1=0.987
  per-category accuracy:
    fake_actor   base=0.500 ft=0.950
    fake_cve     base=0.400 ft=1.000
    real_actor   base=1.000 ft=1.000
    real_cve     base=1.000 ft=1.000


In [9]:
from rouge_score import rouge_scorer
import bert_score
import numpy as np

gold_asst    = [get_asst_msg(r) for r in test_sample]
test_ft      = state["test_ft"]
test_base    = state["test_base"]
test_prompts = [get_user_msg(r) for r in test_sample]

print("\n=== NER F1  ===")
def micro_prf(pred, gold):
    tp = fp = fn = 0
    for p, g in zip(pred, gold):
        ps, gs = extract_entities(p), extract_entities(g)
        tp += len(ps & gs); fp += len(ps - gs); fn += len(gs - ps)
    pr = tp / (tp + fp) if tp + fp else 0
    rc = tp / (tp + fn) if tp + fn else 0
    return {"precision": pr, "recall": rc,
            "f1": 2*pr*rc/(pr+rc) if pr+rc else 0, "tp": tp, "fp": fp, "fn": fn}
ner_base, ner_ft = micro_prf(test_base, gold_asst), micro_prf(test_ft, gold_asst)
for k, n in [("base", ner_base), ("ft", ner_ft)]:
    print(f"  {k.upper():5s} P={n['precision']:.3f} R={n['recall']:.3f} F1={n['f1']:.3f}")

print("\n=== BERTScore ===")
_, _, bs_base = bert_score.score(test_base, gold_asst, lang="en", verbose=False, batch_size=16)
_, _, bs_ft   = bert_score.score(test_ft,   gold_asst, lang="en", verbose=False, batch_size=16)
bertscore_base, bertscore_ft = float(bs_base.mean()), float(bs_ft.mean())
print(f"  BASE={bertscore_base:.3f}  FT={bertscore_ft:.3f}")

print("\n=== ROUGE-L ===")
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
rouge_base = float(np.mean([scorer.score(g, p)["rougeL"].fmeasure for p, g in zip(test_base, gold_asst)]))
rouge_ft   = float(np.mean([scorer.score(g, p)["rougeL"].fmeasure for p, g in zip(test_ft,   gold_asst)]))
print(f"  BASE={rouge_base:.3f}  FT={rouge_ft:.3f}")

print("\n=== Hallucination rate ===")
def hallucination_rate(outputs, inputs):
    rates = []
    for out, inp in zip(outputs, inputs):
        eo, ei = extract_entities(out), extract_entities(inp)
        if eo:
            rates.append(len(eo - ei) / len(eo))
    return float(np.mean(rates)) if rates else 0.0
hall_base = hallucination_rate(test_base, test_prompts)
hall_ft   = hallucination_rate(test_ft,   test_prompts)
print(f"  BASE={hall_base:.3f}  FT={hall_ft:.3f}")

print("\n" + "=" * 74)
print(" SUMMARY  base vs fine-tune")
print("=" * 74)
print(f"{'Metric':40s} {'BASE':>9s} {'FT':>9s} {'delta':>8s}")
print("-" * 74)
def row(name, b, f, lower=False):
    d = f - b; s = "+" if d >= 0 else ""
    print(f"{name:40s} {b:>9.3f} {f:>9.3f} {s}{d:>7.3f}{'  (lower=better)' if lower else ''}")
row("Honesty F1 (refuse)", honesty["base"]["f1_refuse"], honesty["ft"]["f1_refuse"])
row("Honesty accuracy",     honesty["base"]["accuracy"],  honesty["ft"]["accuracy"])
row("NER F1 (extraction)",  ner_base["f1"], ner_ft["f1"])
row("BERTScore",                        bertscore_base, bertscore_ft)
row("ROUGE-L",                          rouge_base, rouge_ft)
row("Hallucination rate",               hall_base, hall_ft, lower=True)
print("=" * 74)

results = {
    "test_sample_size":   len(test_sample),
    "honesty":            honesty,
    "ner":                {"base": ner_base, "ft": ner_ft},
    "bertscore":          {"base": bertscore_base, "ft": bertscore_ft},
    "rouge_l":            {"base": rouge_base, "ft": rouge_ft},
    "hallucination_rate": {"base": hall_base, "ft": hall_ft},
}
with open(RESULTS_FILE, "w") as f:
    json.dump(results, f, indent=2)


=== NER F1  ===
  BASE  P=0.790 R=0.488 F1=0.604
  FT    P=0.963 R=0.811 F1=0.880

=== BERTScore ===


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  BASE=0.819  FT=0.979

=== ROUGE-L ===
  BASE=0.137  FT=0.887

=== Hallucination rate ===
  BASE=0.240  FT=0.088

 SUMMARY  base vs fine-tune
Metric                                        BASE        FT    delta
--------------------------------------------------------------------------
Honesty F1 (refuse)                          0.621     0.987 +  0.367
Honesty accuracy                             0.725     0.988 +  0.263
NER F1 (extraction)                          0.604     0.880 +  0.277
BERTScore                                    0.819     0.979 +  0.160
ROUGE-L                                      0.137     0.887 +  0.750
Hallucination rate                           0.240     0.088  -0.153  (lower=better)


In [10]:
import random
def _t(s, n=320): s = s.replace("\n", " "); return s if len(s) <= n else s[:n] + " ..."

print("#"*74); print("# PROBE  — one per category"); print("#"*74)
for cat in ["real_actor", "real_cve", "fake_actor", "fake_cve"]:
    i = next((j for j, r in enumerate(probe) if r["category"] == cat), None)
    if i is None: continue
    print(f"\n===== {cat}   (expected: {probe[i]['expected_behavior']}) =====")
    print("INPUT:", _t(probe[i]["user_prompt"]))
    print(f"FT   [{classify_refusal(state['probe_ft'][i])}]:", _t(state["probe_ft"][i]))
    print(f"BASE [{classify_refusal(state['probe_base'][i])}]:", _t(state["probe_base"][i]))

print("\n"); print("#"*74); print("# TEST  — random reports"); print("#"*74)
random.seed(0)
for i in random.sample(range(len(test_sample)), min(10, len(test_sample))):
    print(f"\n===== test #{i}  source={test_sample[i].get('source','?')} =====")
    print("INPUT:   ", _t(test_prompts[i]))
    print("EXPECTED:", _t(gold_asst[i]))
    print("FT:      ", _t(state["test_ft"][i]))
    print("BASE:    ", _t(state["test_base"][i]))

##########################################################################
# PROBE  — one per category
##########################################################################

===== real_actor   (expected: answer) =====
INPUT: [MITRE ATT&CK Group lookup] Name: APT28 (G0007) Aliases: APT28, IRON TWILIGHT, SNAKEMACKEREL, Swallowtail, Group 74, Sednit, Sofacy, Pawn Storm, Fancy Bear, STRONTIUM, Tsar Team, Threat Group-4127, TG-4127, Forest Blizzard, FROZENLAKE, GruesomeLarch Attributed techniques: T1001.001, T1003, T1003.001, T1003.003, T1005,  ...
FT   [answer]: Threat Actor Profile: APT28 (G0007)  Known Aliases: APT28, IRON TWILIGHT, SNAKEMACKEREL, Swallowtail, Group 74, Sednit, Sofacy, Pawn Storm, Fancy Bear, STRONTIUM, Tsar Team, Threat Group-4127, TG-4127, Forest Blizzard, FROZENLAKE, GruesomeLarch  Background: APT28 is a threat group that has been attributed to Russia's G ...
BASE [answer]: **Threat Actor Profile: APT28 (G0007)**  **Group Affiliation:** APT28 is attributed to Rus